# Lecture 9 (In-Class): Applied Regression in Economics — Part II
## Interpretation, Robust SE, OVB, and Diagnostics

**Goal:** move from “running OLS” to “thinking like an applied economist.”

**What you will produce today:**
- 2–3 regression models (different functional forms)
- robust standard errors
- 1 diagnostic plot
- a 3-sentence executive summary


## 0) Setup
Run this cell first.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import statsmodels.api as sm
import statsmodels.formula.api as smf

np.random.seed(4370)
pd.set_option("display.max_columns", 50)

## 1) Create a realistic (synthetic) wage dataset
This dataset is **synthetic** but designed to mimic common patterns in labor econ:
- wages rise with education and experience (diminishing returns)
- union membership increases wages
- unobserved ability affects both education and wages (OVB concept)

Using synthetic data keeps this notebook self-contained (no API keys, no downloads).

In [ ]:
n = 1500

# Unobserved "ability" (confounder): affects education and wages
ability = np.random.normal(0, 1, n)

educ = np.clip(np.round(12 + 2.2*ability + np.random.normal(0, 2.0, n)), 8, 20).astype(int)
exper = np.clip(np.round(np.random.uniform(0, 35, n)), 0, 35).astype(int)
female = np.random.binomial(1, 0.50, n)
union = np.random.binomial(1, 0.18 + 0.02*(educ>=14), n)  # slightly more likely w/ college

# True wage process (log wages)
# (ability is omitted at first to show OVB when we ignore it)
log_wage = (
    1.6
    + 0.075*educ
    + 0.035*exper
    - 0.00055*(exper**2)          # diminishing returns to experience
    + 0.10*union
    - 0.07*female
    + 0.18*ability                # omitted variable (confounder)
    + np.random.normal(0, 0.25, n)
)

wage = np.exp(log_wage)

df = pd.DataFrame({
    "wage": wage,
    "log_wage": np.log(wage),
    "educ": educ,
    "exper": exper,
    "female": female,
    "union": union,
    "ability": ability
})

df.head()

## 2) Quick EDA
Look at distributions and simple relationships.

In [ ]:
df.describe().T

In [ ]:
plt.figure()
plt.hist(df["wage"], bins=40)
plt.title("Wage distribution (right-skewed)")
plt.xlabel("wage")
plt.ylabel("count")
plt.show()

plt.figure()
plt.hist(df["log_wage"], bins=40)
plt.title("Log wage distribution (more symmetric)")
plt.xlabel("log(wage)")
plt.ylabel("count")
plt.show()

## 3) Model 1: Level–Level OLS
Estimate: wage on education and experience.

**Interpretation prompt:** What are the *units* of wage? What does a 1-unit change in `educ` mean?

In [ ]:
m1 = smf.ols("wage ~ educ + exper", data=df).fit()
m1.summary()

### Robust standard errors
Now compute robust SE (HC1). Compare p-values and confidence intervals.

In [ ]:
m1_rob = smf.ols("wage ~ educ + exper", data=df).fit(cov_type="HC1")
m1_rob.summary()

## 4) Model 2: Log–Level (common in labor econ)
Estimate: log(wage) on education and experience.

**Interpretation prompt:** If `educ` coefficient is 0.07, what does that mean in percent terms?

In [ ]:
m2 = smf.ols("log_wage ~ educ + exper", data=df).fit(cov_type="HC1")
m2.summary()

### Convert log coefficient to percent (more precise)
For a coefficient b: percent change ≈ 100*(exp(b)-1)

In [ ]:
b_educ = m2.params["educ"]
pct = 100*(np.exp(b_educ)-1)
b_educ, pct

## 5) Model 3: Add controls (union, female)
See how coefficients change when you add economically relevant controls.

**Prompt:** Does the education coefficient change? Why might it?

In [ ]:
m3 = smf.ols("log_wage ~ educ + exper + union + female", data=df).fit(cov_type="HC1")
m3.summary()

## 6) Omitted Variable Bias Demo
We *intentionally* omitted `ability` (unobserved in real life).
Now include it to show what would happen if we could observe it.

**Key question:** How does the estimated return to education change?

In [ ]:
m4 = smf.ols("log_wage ~ educ + exper + union + female + ability", data=df).fit(cov_type="HC1")
m4.summary()

### Compare education coefficients across models

In [ ]:
compare = pd.DataFrame({
    "model": ["m2: log_wage ~ educ + exper",
              "m3: + union + female",
              "m4: + ability (confounder observed)"],
    "educ_coef": [m2.params["educ"], m3.params["educ"], m4.params["educ"]],
    "educ_se_robust": [m2.bse["educ"], m3.bse["educ"], m4.bse["educ"]],
})
compare

## 7) Diagnostics
We’ll do two practical checks:
1) Residuals vs fitted (nonlinearity / heteroskedasticity clues)
2) Influence (Cook’s distance)

Use model m3 as your baseline.

In [ ]:
# Residuals vs fitted
fitted = m3.fittedvalues
resid = m3.resid

plt.figure()
plt.scatter(fitted, resid, s=10)
plt.axhline(0)
plt.title("Residuals vs Fitted (Model m3)")
plt.xlabel("Fitted values")
plt.ylabel("Residuals")
plt.show()

In [ ]:
infl = m3.get_influence()
cooks = infl.cooks_distance[0]

plt.figure()
plt.stem(np.arange(len(cooks)), cooks, markerfmt=",", basefmt=" ")
plt.title("Cook's distance (Model m3)")
plt.xlabel("Observation index")
plt.ylabel("Cook's D")
plt.show()

# Show top 10 most influential observations
top_idx = np.argsort(cooks)[-10:][::-1]
df.loc[top_idx, ["wage","educ","exper","union","female"]].assign(cooksD=cooks[top_idx])

## 8) Executive Summary (write in markdown)
Write **3 sentences** as if you are briefing a manager or policy analyst.

Include:
1) the main relationship (education → wages)
2) an uncertainty statement (robust SE / not necessarily causal)
3) one limitation (OVB / omitted factors / synthetic data)

**Replace the bracketed text below with your own writing.**

**Executive Summary:**

1. [Sentence 1]

2. [Sentence 2]

3. [Sentence 3]


## 9) Extension (if time)
1) Try a nonlinear experience term: add `I(exper**2)` in the formula.
2) Compare models using AIC/BIC.
3) Create a clean regression table (optional).

In [ ]:
m5 = smf.ols("log_wage ~ educ + exper + I(exper**2) + union + female", data=df).fit(cov_type="HC1")
m5.summary()